# Data Science for Business - Fair ML for Credit Scoring


## Initialize notebook
Load required packages.

In [ ]:
# Install packages that are not already installed on Colab
# !pip install aif360

In [ ]:
import warnings
warnings.simplefilter('ignore')

import numpy as np
import pandas as pd

import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, f1_score

from aif360.sklearn.datasets import fetch_german
from aif360.sklearn.metrics import disparate_impact_ratio, equal_opportunity_difference
from aif360.sklearn.preprocessing import Reweighing
from aif360.sklearn.postprocessing import RejectOptionClassifier, RejectOptionClassifierCV


## Problem description

In this notebook we will explore fairness in machine learning models for credit scoring using the German Credit dataset (https://archive.ics.uci.edu/dataset/144/statlog+german+credit+data). The goal is to predict whether an individual has good or bad credit risk while ensuring that the model does not discriminate based on sensitive attributes such as gender or age.


## Load data

First, we load the German Credit data from the AIF360 package. For more information, see: https://aif360.readthedocs.io/en/stable/modules/generated/aif360.sklearn.datasets.fetch_german.html

In [ ]:
data = fetch_german(numeric_only=True)

In [ ]:
data.X

In [ ]:
data.y

## Train classifier

Next, we train a standard classifier using sklearn without any fairness constraints.

In [ ]:
X_train = data.X[:800]
y_train = data.y[:800]

X_test = data.X[800:]
y_test = data.y[800:]

In [ ]:
classifier_logit = LogisticRegression().fit(X_train, y_train)

In [ ]:
y_pred = classifier_logit.predict(X_test)
y_proba = classifier_logit.predict_proba(X_test)

In [ ]:
f1_score(y_test, y_pred)

## Compute fairness metrics

### Equal Opportunity Difference

Here we compute the Equal Opportunity Difference metric before applying any fairness mitigation techniques. The negative value indicates implies higher benefit for the privileged group. However, the value is relatively close to zero, indicating only a small disparity between groups.

In [ ]:
eod = equal_opportunity_difference(y_test, y_pred, prot_attr='sex', priv_group=1)
print(eod)

sns.barplot(x=['Equal Opportunity Difference'], y=[eod])
plt.axhline(y=0.0, color='green', linestyle='--')
plt.fill_between(x=[-0.5, 0.5], y1=-0.1, y2=0.1, color='green', alpha=0.2)
plt.title('Equal Opportunity Difference for Sex')
plt.ylim(-1.0, 1.0)
plt.show()

### Disparate Impact Ratio

Similarly, we compute the Disparate Impact Ratio metric. Again, the value less than 1 indicates implies higher benefit for the privileged group. However, the value is within the bounds of the "four-fifths rule" (i.e., between 0.8 and 1.25), suggesting that the disparity is not substantial.

In [ ]:
di = disparate_impact_ratio(y_test, y_pred, prot_attr='sex', priv_group=1)
print(di)

sns.barplot(x=['Disparate Impact Ratio'], y=[di])
plt.axhline(y=1.0, color='green', linestyle='--')
plt.fill_between(x=[-0.5, 0.5], y1=0.8, y2=1.25, color='green', alpha=0.2)
plt.title('Disparate Impact Ratio for Sex')
plt.ylim(0, 2)
plt.show()

## Bias mitigation algorithms

Altough the fairness metrics indicate only a small disparity, we will still apply two common bias mitigation techniques to see their effect on fairness and accuracy.

### Pre-processing algorithms: Reweighing

In [ ]:
rw_preprocessor = Reweighing(prot_attr='sex')
new_weights = rw_preprocessor.fit_transform(X_train, y_train)

Display the the weights assigned to each training observation.

In [ ]:
new_weights

Retrain the classifier using the computed sample weights.

In [ ]:
classifier_logit_rw = LogisticRegression().fit(X_train, y_train, sample_weight=new_weights[1])

In [ ]:
y_pred_rw = classifier_logit_rw.predict(X_test)

Compute the Equal Opportunity Difference and Disparate Impact Ratio metrics after applying the Reweighing techniques. In both cases we now see higher benefit for the unprivileged group (i.e., EOD larger than 0 and DIR larger than 1).

In [ ]:
eod_rw = equal_opportunity_difference(y_test, y_pred_rw, prot_attr='sex', priv_group=1)
print(eod_rw)

sns.barplot(x=['Equal Opportunity Difference'], y=[eod_rw])
plt.axhline(y=0.0, color='green', linestyle='--')
plt.fill_between(x=[-0.5, 0.5], y1=-0.1, y2=0.1, color='green', alpha=0.2)
plt.title('Equal Opportunity Difference for Sex')
plt.ylim(-1.0, 1.0)
plt.show()


In [ ]:
di_rw = disparate_impact_ratio(y_test, y_pred_rw, prot_attr='sex', priv_group=1)
print(di_rw)

sns.barplot(x=['Disparate Impact Ratio'], y=[di_rw])
plt.axhline(y=1.0, color='green', linestyle='--')
plt.fill_between(x=[-0.5, 0.5], y1=0.8, y2=1.25, color='green', alpha=0.2)
plt.title('Disparate Impact Ratio for Sex')
plt.ylim(0, 2)
plt.show()

Surprisingly, the accuracy has even improved after applying the Reweighing technique.

In [ ]:
f1_score(y_test, y_pred_rw)

### Post-processing algorithms: Reject Option Classification

For ROC, we need to choose the margin in which we want to change the labels. Here, we choose a margin of 0.02 around the decision boundary (i.e., 0.5).

In [ ]:
roc_postprocessor = RejectOptionClassifier(prot_attr='sex', threshold=0.5, margin=0.02)

As ROC is a post-processing technique, we do not need to retrain the classifier. Instead, we directly apply ROC to the predictions of the original classifier.

In [ ]:
roc_postprocessor.fit(X_train, y_train)

In [ ]:
y_preds_roc = roc_postprocessor.predict(pd.DataFrame(y_proba, index=y_test.index))

Check fairness metrics after Reject Option Classification. Again, we see higher benefit for the unprivileged group after bias mitigation. And compared to Reweighing, ROC procuded more fair results (i.e., EOD closer to 0 and DIR closer to 1).

In [ ]:
eod_roc = equal_opportunity_difference(y_test, y_preds_roc, prot_attr='sex', priv_group=1)
print(eod_roc)

sns.barplot(x=['Equal Opportunity Difference'], y=[eod_roc])
plt.axhline(y=0.0, color='green', linestyle='--')
plt.fill_between(x=[-0.5, 0.5], y1=-0.1, y2=0.1, color='green', alpha=0.2)
plt.title('Equal Opportunity Difference for Sex')
plt.ylim(-1.0, 1.0)
plt.show()


In [ ]:
di_roc = disparate_impact_ratio(y_test, y_preds_roc, prot_attr='sex', priv_group=1)
print(di_roc)

sns.barplot(x=['Disparate Impact Ratio'], y=[di_roc])
plt.axhline(y=1.0, color='green', linestyle='--')
plt.fill_between(x=[-0.5, 0.5], y1=0.8, y2=1.25, color='green', alpha=0.2)
plt.title('Disparate Impact Ratio for Sex')
plt.ylim(0, 2)
plt.show()

And the accuracy did not change much compared to the original classifier.

In [ ]:
f1_score(y_test, y_preds_roc)